# Day 26 — The Uniform Distribution
### Statistics & Probability · Module 2 · Topic 2.6

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours · 90 min concepts + 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | **Discrete uniform** — a die, and the two formulas | 15 min |
| 2 | **Continuous uniform** — the rectangle, and why area is everything | 30 min |
| 3 | Is it really uniform? Testing it, and two famous failures | 20 min |
| 4 | **The foundation** — every random number starts as U(0,1) | 20 min |
| 5 | Where it leads — one uniform is flat, twelve are a bell | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

---

## The step we take today

The uniform looks like the easy session. A flat line — what is there to learn?

Two things, and both of them matter more than the distribution itself.

**First**, this is where *probability becomes area*. For a discrete distribution you read the
probability off the height of a bar. For a continuous one you cannot, and the replacement —
area under the curve — is the idea that makes every continuous distribution work, including
yesterday's normal. It also produces a result that sounds wrong the first time you hear it:
the probability of any exact value is **zero**.

**Second**, the uniform is not one distribution among several. It is the one your computer
actually produces. Every normal, binomial and Poisson sample you have drawn this term was a
stream of uniform numbers with a transformation applied. Section 4 shows you the transformation.

### Three claims to hold me to

1. The probability of any exact value is **zero**.
2. The height of the curve can be **bigger than 1**.
3. Every random number your computer makes **begins here**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

plt.rcParams["figure.figsize"] = (8, 3.2)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 1. Discrete uniform

n outcomes, each with probability 1/n. That is the whole thing.

In [ ]:
die = stats.randint(1, 7)          # faces 1 to 6

ks = np.arange(1, 7)
print("pmf:", die.pmf(ks))
print()
print(f"mean = {die.mean()}      = (a+b)/2 = {(1+6)/2}")
print(f"var  = {die.var():.4f}  = (n^2-1)/12 = {(36-1)/12:.4f}")
print(f"sd   = {die.std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(ks, die.pmf(ks), color="#3E7CB1", width=0.62)
ax.axhline(1/6, color="#E8674C", ls="--", lw=1.8)
for k in ks:
    ax.text(k, 1/6 + 0.006, "1/6", ha="center", fontweight="bold")
ax.set_xticks(ks); ax.set_ylim(0, 0.21)
ax.set_xlabel("Face of the die"); ax.set_ylabel("Probability")
ax.set_title("Discrete uniform: every bar the same height")
plt.show()

## ⚠️ The bounds trap, three ways

Three libraries, three conventions, and none of them raises an error when you get it wrong.

In [ ]:
import random

print("scipy  stats.randint(1, 7)  -> 1..6   (high EXCLUSIVE)")
print("   pmf(6) =", stats.randint(1, 7).pmf(6))
print("   pmf(7) =", stats.randint(1, 7).pmf(7), " <- 7 is excluded")
print()
print("the bug:  stats.randint(1, 6)  is a FIVE-sided die")
print("   pmf(6) =", stats.randint(1, 6).pmf(6), " <- silently impossible")
print("   mean   =", stats.randint(1, 6).mean(), "instead of 3.5")
print()
rng = np.random.default_rng(1)
print("numpy  rng.integers(1, 7, 10) ->", rng.integers(1, 7, 10), " (high exclusive)")
random.seed(1)
print("python random.randint(1, 6)   ->", [random.randint(1, 6) for _ in range(10)],
      " (high INCLUSIVE)")

`numpy` and `scipy` exclude the top value. Python's own `random` module includes it. Nothing
warns you; you just quietly build a five-sided die and every probability shifts by a sixth.

## A mean the outcome can never equal

The mean of a die is 3.5, and no roll will ever show 3.5. That is not a mistake — a mean is a
long-run average, not a possible outcome. Same idea as "2.4 children per family".

In [ ]:
rng = np.random.default_rng(26)
rolls = rng.integers(1, 7, 5000)
running = np.cumsum(rolls) / np.arange(1, 5001)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(running, color="#3E7CB1", lw=1.2)
ax.axhline(3.5, color="#E8674C", ls="--", lw=2)
ax.text(5050, 3.5, "3.5", va="center", color="#E8674C", fontweight="bold")
ax.set_xlabel("number of rolls"); ax.set_ylabel("running average")
ax.set_ylim(2.5, 4.5); ax.set_title("It heads for 3.5 without ever landing on it")
plt.show()

print("after 5000 rolls:", round(running[-1], 4))

---
# 2. Continuous uniform

## 2.1 The bridge: what happens as outcomes get closer together

With 6 outcomes each bar is 1/6 tall. With 100, each is 1/100. Keep going and every
*individual* probability goes to zero — but the total stays 1, because there are more of them.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 2.7))
for ax, n in zip(axes[:3], [6, 20, 100]):
    e = np.linspace(0, 10, n + 1)
    ax.bar((e[:-1] + e[1:]) / 2, np.full(n, 1/n), width=(10/n) * 0.88, color="#3E7CB1")
    ax.set_title(f"{n} values, each {1/n:.3f}", fontsize=10)
axes[3].fill_between([0, 10], 0, 0.1, color="#2FA98C", alpha=0.8)
axes[3].set_ylim(0, 0.13); axes[3].set_title("the limit: a rectangle", fontsize=10)
plt.tight_layout(); plt.show()

In the limit there are no bars at all. A value can be 3, or 3.1, or 3.14159 — infinitely many
possibilities. Asking for the probability of *one of them* stops making sense.

## ⭐ 2.2 The key idea: probability is AREA

The height of the curve is no longer a probability. The **area under it** is.

In [ ]:
u = stats.uniform(0, 10)          # NOTE: loc=0, scale=10  ->  U(0, 10)

print(f"height of the curve anywhere inside = {u.pdf(3)}   = 1/(b-a) = {1/10}")
print()
print(f"P(X < 3)       = {u.cdf(3):.4f}      area of a 3-wide strip")
print(f"P(3 < X < 7)   = {u.cdf(7) - u.cdf(3):.4f}      4 wide x 0.1 tall")
print(f"P(X > 8)       = {1 - u.cdf(8):.4f}")
print()
print(f"total area     = {u.cdf(10) - u.cdf(0):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))

ax = axes[0]
ax.fill_between([0, 10], 0, 0.1, color="#D8E1EB")
ax.fill_between([3, 7], 0, 0.1, color="#2FA98C", alpha=0.85)
ax.text(5, 0.04, "width 4\nx height 0.1\n= 0.4", ha="center", color="white",
        fontweight="bold")
ax.set_ylim(0, 0.13); ax.set_title("P(3 < X < 7) = 0.40")

ax = axes[1]
ax.fill_between([0, 10], 0, 0.1, color="#D8E1EB")
ax.plot([5, 5], [0, 0.1], color="#E8674C", lw=3)
ax.text(5.3, 0.05, "a line has no width\nso AREA = 0", color="#E8674C", fontweight="bold")
ax.set_ylim(0, 0.13); ax.set_title("P(X = exactly 5) = 0")
for ax in axes:
    ax.set_xlabel("value"); ax.set_ylabel("density")
plt.tight_layout(); plt.show()

print("P(X == 5) =", u.cdf(5) - u.cdf(5))

## 2.3 "Probability zero" does not mean "impossible"

This is the fair objection: *if every value has probability 0, how does anything happen?*

**The resolution:** you never observe a value to infinite precision. Your instrument said 5,
which means somewhere in 4.995 to 5.005 — and **that** has a real, non-zero probability.

In [ ]:
for width in [1, 0.1, 0.01, 0.001, 0.0]:
    lo, hi = 5 - width/2, 5 + width/2
    print(f"P(5 +/- {width/2:<7}) = {u.cdf(hi) - u.cdf(lo):.6f}")

print()
print("The probability shrinks with the width and hits 0 only when the width does.")
print("Every real observation is a range. 'Exactly 5' is an idealisation.")

**The practical rule:** for discrete data, "what is the probability of exactly 3?" has a real
answer. For continuous data, never ask it — ask about a range. If your code returns a
probability of exactly `0.0` for a continuous variable, check whether you asked for a point.

## ⚠️ 2.4 A density is not a probability — it can exceed 1

In [ ]:
print(f"{'distribution':<14} {'height':>8} {'area':>8}")
for b in [10, 1, 0.5, 0.1, 0.01]:
    d = stats.uniform(0, b)
    print(f"{'U(0, ' + format(b, 'g') + ')':<14} {d.pdf(b/2):>8.4g} {d.cdf(b):>8.1f}")

Squeeze the same amount of probability into a narrower range and the height has to go up. At
U(0, 0.01) the density is **100**. Nothing is broken.

A density is probability **per unit of x** — like a speed, or a price per kilo. A price of
£300 per kilo is not a contradiction; you just have not bought a kilo.

So `pdf()` never returns a probability. Use `cdf()` for that.

## 2.5 The cdf is a straight ramp

In [ ]:
xs = np.linspace(-1, 11, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(xs, u.pdf(xs), color="#2FA98C", lw=2.5)
axes[0].fill_between([0, 10], 0, 0.1, color="#2FA98C", alpha=0.3)
axes[0].set_title("pdf - flat"); axes[0].set_ylabel("density")
axes[1].plot(xs, u.cdf(xs), color="#3E7CB1", lw=2.5)
axes[1].set_title("cdf - a straight ramp"); axes[1].set_ylabel("P(X < value)")
for ax in axes:
    ax.set_xlabel("value")
plt.tight_layout(); plt.show()

The uniform is the **only** distribution whose cdf is a straight line. That is worth noticing
rather than generalising — the normal's cdf is an S, the Poisson's is a staircase.

## 2.6 Worked example: waiting for the bus

Buses every 15 minutes. You turn up without checking the timetable, so your arrival is equally
likely at any point in the gap. Your wait is **U(0, 15)**.

In [ ]:
bus = stats.uniform(0, 15)        # loc=0, scale=15

print(f"mean wait     = {bus.mean()} min        = (a+b)/2")
print(f"sd            = {bus.std():.4f} min     = 15/sqrt(12) = {15/np.sqrt(12):.4f}")
print()
print(f"P(wait < 5)   = {bus.cdf(5):.4f}")
print(f"P(wait > 10)  = {1 - bus.cdf(10):.4f}")
print(f"P(5 < w < 10) = {bus.cdf(10) - bus.cdf(5):.4f}")
print()
print("A third under 5 minutes, a third over 10.")
print("'An average wait of 7.5 minutes' hides both ends.")

## ⚠️ 2.7 The scipy trap: loc and SCALE, not the two endpoints

This one will catch you at least once.

In [ ]:
#  you probably meant U(5, 10)
wrong = stats.uniform(5, 10)
right = stats.uniform(5, 5)       # loc=5, scale=5  ->  U(5, 10)

print("stats.uniform(5, 10) spans", wrong.ppf([0, 1]), " <- that is U(5, 15)")
print("stats.uniform(5,  5) spans", right.ppf([0, 1]), " <- U(5, 10), correct")
print()
print(f"mean: wrong {wrong.mean()}   right {right.mean()}")
print()
print("RULE:  U(a, b)  is  stats.uniform(a, b - a)")

The second argument is the **width**. Get it wrong and you get the right lower bound, a
silently wrong upper one, and a mean that is out by half the width.

Mean and variance for the continuous case:

- mean = (a + b) / 2
- variance = (b − a)² / 12  →  **sd of U(0,1) is 1/√12 = 0.2887**

Remember that 0.2887. It reappears the moment anyone adds uniforms together.

---
# 3. Is it really uniform?

Uniform is a genuine model in some places and a lazy default in others.

| ✅ Genuinely uniform | ❌ Assumed, and wrong |
|---|---|
| **Rounding error** — provably U(−0.05, 0.05) when rounding to 0.1 | **Exam marks across 0–100** — marks cluster |
| **A fair mechanism** — a die, a shuffled deck, a lottery | **Birthdays across the year** — September is busier |
| **Arrival within a known gap** — no timetable | **Leading digits** — nowhere near uniform |
| **"All I know is the bounds"** — the honest model | **"Customers arrive uniformly"** — lunchtime exists |

The pattern: uniform is right when something *makes* it uniform. It is wrong whenever it was
chosen because nobody looked.

## 3.1 Rounding error really is uniform

In [ ]:
rng = np.random.default_rng(2603)
prices = rng.random(100_000) * 100
err = prices - np.round(prices, 1)

print(f"range     {err.min():.4f} to {err.max():.4f}   (exactly -0.05 to 0.05)")
print(f"mean      {err.mean():.5f}   theory 0")
print(f"sd        {err.std():.5f}   theory 0.1/sqrt(12) = {0.1/np.sqrt(12):.5f}")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.hist(err, bins=50, color="#2FA98C")
ax.set_title("Rounding error - uniform as a matter of arithmetic, not approximation")
plt.show()

## 3.2 Testing a die

A fair die does **not** give 100 of each in 600 rolls. So "the counts are uneven" is not
evidence of anything. The chi-square test asks *how far off is too far*.

In [ ]:
rng = np.random.default_rng(2601)
fair = np.bincount(rng.integers(1, 7, 600), minlength=7)[1:]

rng2 = np.random.default_rng(2601)
loaded_p = np.array([1, 1, 1, 1, 1, 2]) / 7     # 6 comes up twice as often
_ = rng2.choice(np.arange(1, 7), 60,  p=loaded_p)
_ = rng2.choice(np.arange(1, 7), 200, p=loaded_p)
loaded = np.bincount(rng2.choice(np.arange(1, 7), 600, p=loaded_p), minlength=7)[1:]

for name, counts in [("fair  ", fair), ("loaded", loaded)]:
    p = stats.chisquare(counts).pvalue
    verdict = "not uniform" if p < 0.05 else "no evidence against uniform"
    print(f"{name} {counts}  spread {counts.min()}-{counts.max()}  p = {p:.2e}  {verdict}")

The fair die ranged from 85 to 119 and the test still says there is nothing wrong. That spread
is normal.

## ⚠️ 3.3 How much data would you need?

A small p-value says "too uneven to be chance". A large one does **not** prove fairness — it
may just mean you did not roll enough.

In [ ]:
rng = np.random.default_rng(777)
print(f"{'rolls':>7} {'caught it':>12}")
power = []
for n in [40, 60, 80, 100, 150, 200, 300, 400, 600]:
    hits = sum(stats.chisquare(
                   np.bincount(rng.choice(np.arange(1, 7), n, p=loaded_p), minlength=7)[1:]
               ).pvalue < 0.05
               for _ in range(600))
    power.append((n, 100 * hits / 600))
    print(f"{n:>7} {100*hits/600:>11.0f}%")

In [ ]:
ns, pw = zip(*power)
fig, ax = plt.subplots(figsize=(8, 3))
ax.axhspan(0, 80, color="#FDECE8")
ax.plot(ns, pw, color="#3E7CB1", lw=2.5, marker="o")
ax.axhline(80, color="#152238", ls="--", lw=1.8)
ax.set_xlabel("Number of rolls"); ax.set_ylabel("how often you CATCH it (%)")
ax.set_ylim(0, 105)
ax.set_title("A die that favours 6 twice over - how often does the test spot it?")
plt.show()

This die is blatantly crooked — 28.6% instead of 16.7% — and **60 rolls catch it only 40% of
the time**. You would miss it more often than you found it.

Which is the same lesson as yesterday's A/B test: *"I tested it and found nothing"* means
nothing until you say how much data you had.

## 3.4 Continuous: same range, same mean, not uniform

In [ ]:
rng = np.random.default_rng(2602)
good = rng.random(2000)
bad  = rng.beta(2, 2, 2000)       # bell-ish on [0,1]

print(f"{'':<8} {'min':>6} {'max':>6} {'mean':>8} {'sd':>8} {'KS p':>12}")
for name, d in [("uniform", good), ("beta   ", bad)]:
    print(f"{name:<8} {d.min():>6.3f} {d.max():>6.3f} {d.mean():>8.4f} "
          f"{d.std():>8.4f} {stats.kstest(d, 'uniform').pvalue:>12.2e}")
print()
print(f"theory sd for U(0,1) = {1/np.sqrt(12):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, d, t, c in [(axes[0], good, "Really uniform", "#2FA98C"),
                    (axes[1], bad,  "Beta(2,2) - NOT uniform", "#E8674C")]:
    ax.hist(d, bins=20, color=c)
    ax.axhline(len(d)/20, color="#152238", ls="--", lw=1.8)
    ax.set_title(t); ax.set_xlabel("value")
plt.tight_layout(); plt.show()

Both run 0 to 1. Both average about 0.51. **Checking the bounds and the mean tells you
nothing** — the shape in between is the whole question. The sd is the quick tell (0.294 vs
0.227), and a histogram is faster than any test.

## 3.5 Benford's law — the most famous thing that is not uniform

In [ ]:
benford = np.log10(1 + 1/np.arange(1, 10))

rng = np.random.default_rng(26)
vals = 10 ** (rng.random(100_000) * 6)        # spread over 6 orders of magnitude
lead = np.array([int(f"{v:.10e}"[0]) for v in vals])
observed = np.bincount(lead, minlength=10)[1:] / len(lead)

print(f"{'digit':>6} {'observed':>10} {'Benford':>10} {'uniform':>10}")
for d in range(1, 10):
    print(f"{d:>6} {observed[d-1]:>10.4f} {benford[d-1]:>10.4f} {1/9:>10.4f}")
print()
print(f"1 appears {benford[0]/(1/9):.2f}x as often as uniform predicts")

In [ ]:
d = np.arange(1, 10)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(d - 0.2, benford, width=0.4, color="#3E7CB1", label="what really happens")
ax.bar(d + 0.2, np.full(9, 1/9), width=0.4, color="#9AA7B8", label="what 'random' would give")
ax.set_xticks(d); ax.set_xlabel("leading digit"); ax.set_ylabel("proportion")
ax.legend(frameon=False)
ax.set_title("Benford's law")
plt.show()

**Why:** real quantities spread across orders of magnitude. Something growing from 100 to 200
doubles; from 900 to 1000 it rises 11%. So a value spends far longer with a leading 1 than a
leading 9.

**Why it matters:** this is used to detect fraud. People inventing numbers spread their leading
digits roughly evenly, because that is what "random" feels like. Real accounts do not. A
dataset that is *suspiciously uniform* is the red flag.

---
# 4. The foundation — every random number starts as U(0,1)

## 4.1 The claim

Underneath, your computer does exactly one random thing:

```python
rng.random()   ->   0.7340815...
```

Every other distribution you have called this term is a **transformation** applied to a stream
of these. If that is true, you should be able to build a normal distribution by hand without
ever calling `rng.normal`.

## 4.2 Inverse transform sampling

Pick a number between 0 and 1. Treat it as a position on the **vertical** axis of the cdf. Go
across to the curve, then drop down. The value you land on is your sample.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
xs = np.linspace(-3.4, 3.4, 400)
ax.plot(xs, stats.norm.cdf(xs), color="#1F3A5F", lw=2.5, zorder=3)
for uu, col in [(0.80, "#E8674C"), (0.50, "#C98A00"), (0.15, "#2FA98C")]:
    x = stats.norm.ppf(uu)
    ax.plot([-3.6, x], [uu, uu], color=col, ls="--", lw=1.8)
    ax.plot([x, x], [uu, -0.04], color=col, ls="--", lw=1.8)
    ax.plot([x], [uu], "o", color=col, ms=8, zorder=4)
    ax.text(-3.5, uu + 0.03, f"u = {uu:.2f}", color=col, fontweight="bold", fontsize=10)
    ax.text(x, -0.1, f"{x:+.2f}", ha="center", color=col, fontweight="bold", fontsize=10)
ax.set_ylim(-0.15, 1.05)
ax.set_xlabel("the normal value you get out")
ax.set_ylabel("the uniform number you put in")
ax.set_title("Inverse transform sampling")
plt.show()

**Why it works:** where the cdf is **steep**, a small slice of the vertical axis maps to a
narrow range of x, so values pile up there. Where it is **flat**, the same slice spreads across
a wide range, so values are rare.

Steep means likely. That is exactly what a density is.

## 4.3 It actually works

In [ ]:
rng = np.random.default_rng(26)
u01 = rng.random(200_000)          #  ONE source of randomness

normal = stats.norm.ppf(u01)                 # inverse cdf of the normal
expon  = -np.log(1 - u01) / 0.5              # inverse cdf of Exp(rate 0.5), by hand

print(f"normal      : mean {normal.mean():+.4f}  sd {normal.std():.4f}   (target 0, 1)")
print(f"exponential : mean {expon.mean():.4f}                    (target 2)")
print(f"scipy expon : mean {stats.expon(scale=2).mean()}")
print()
print("no rng.normal, no rng.exponential - just rng.random() and a formula")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.9))
axes[0].hist(u01, bins=40, color="#FFC145");  axes[0].set_title("1. start with U(0,1)")
axes[1].hist(normal, bins=60, color="#3E7CB1"); axes[1].set_title("2. norm.ppf -> Normal")
axes[2].hist(expon, bins=60, color="#2FA98C");  axes[2].set_title("3. -ln(1-u)/0.5 -> Exponential")
plt.tight_layout(); plt.show()

## 4.4 The discrete version: build a loaded die

For a discrete distribution the "inverse cdf" is just *which band did the number land in*.

In [ ]:
rng = np.random.default_rng(26)
probs = np.array([.1, .1, .1, .1, .1, .5])   # 6 comes up half the time

cum = np.cumsum(probs)
print("cumulative:", cum)
print()

u = rng.random(200_000)
rolls = np.searchsorted(cum, u) + 1          # +1 because faces start at 1

freq = np.bincount(rolls, minlength=7)[1:] / len(rolls)
print("asked for :", probs)
print("got       :", freq.round(4))
print()
print("This is roughly what rng.choice(p=...) does internally.")

## ⚠️ 4.5 None of this is actually random

In [ ]:
print(np.random.default_rng(42).random(3))
print(np.random.default_rng(42).random(3))
print("same seed, same numbers, every time")

It is a deterministic formula producing a sequence that **passes** the uniformity tests.
Statistically uniform, not unpredictable.

Which is a feature:

- Your results reproduce — yours and everyone else's
- A failing test fails the same way twice
- A reviewer can rerun your notebook and get your numbers
- Every figure in this course is seeded for exactly this reason

**But:** never use `numpy.random` or Python's `random` for anything security-related —
passwords, tokens, keys. Given enough output the sequence can be worked backwards and every
future value predicted. Use the `secrets` module. It is slower, and it is not reproducible,
and both of those are the point.

In [ ]:
import secrets
print("a cryptographically secure token:", secrets.token_hex(16))
print("run it again and it will be different - there is no seed")

---
# 5. Where it leads

One uniform is flat. What happens if you add two together?

In [ ]:
rng = np.random.default_rng(2606)

print(f"{'n':>3} {'mean':>8} {'sd':>8} {'skew':>9} {'kurtosis':>10}")
for n in [1, 2, 3, 12]:
    s = rng.random((200_000, n)).sum(axis=1)
    print(f"{n:>3} {s.mean():>8.4f} {s.std():>8.4f} "
          f"{stats.skew(s):>+9.4f} {stats.kurtosis(s):>+10.4f}")
print()
print("A normal distribution has skew 0 and kurtosis 0.")
print("Theory for the sum of 12 U(0,1): mean 6, variance 12/12 = 1, so sd exactly 1.")

In [ ]:
rng = np.random.default_rng(2606)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.9))
for ax, n in zip(axes, [1, 2, 3, 12]):
    s = rng.random((120_000, n)).sum(axis=1)
    ax.hist(s, bins=70, color="#3E7CB1", density=True)
    g = np.linspace(s.min(), s.max(), 300)
    ax.plot(g, stats.norm.pdf(g, n/2, np.sqrt(n/12)), color="#152238", lw=2)
    ax.set_title(f"sum of {n}", fontsize=10); ax.set_yticks([])
plt.tight_layout(); plt.show()

Nothing about a flat line suggests a bell curve. But two uniforms make a **triangle** — totals
near the middle have many ways to happen and totals near the ends have few — three start to
curve, and twelve are indistinguishable from a normal.

That is the **third** time this has happened:

- the binomial did it as n grew (Day 25)
- the Poisson did it as λ grew (Day 25)
- and now the flattest distribution of all does it too

Day 27 explains why it is not a coincidence.

---

## Module 2 so far — four distributions

| | Normal | Binomial | Poisson | Uniform |
|---|---|---|---|---|
| Day | 24 | 25 | 25 | 26 |
| Type | continuous | discrete | discrete | either |
| Answers | how big? | how many out of n? | how many per interval? | equally likely |
| Parameters | μ, σ | n, p | λ | a, b |
| Mean | μ | n × p | λ | (a + b) / 2 |
| Variance | σ² | np(1−p) | λ | (b − a)² / 12 |
| Shape | bell | peak at np | right-skewed | flat |
| scipy | `norm` | `binom` | `poisson` | `uniform` / `randint` |

---

# Debug corner — eight uniform mistakes

| You did | What goes wrong | Fix |
|---|---|---|
| `uniform(5, 10)` for U(5, 10) | scipy takes loc and **width**: you got U(5, 15) | `uniform(5, 5)`, or `uniform(a, b-a)` |
| read `pdf()` as a probability | it can exceed 1 and is not a probability | `cdf` for probabilities, `pdf` for density |
| asked P(X = 4.7) on continuous data | always exactly 0, silently | ask about a range |
| `randint(1, 6)` for a six-sided die | high is **exclusive**: you built a 5-sided die | `randint(1, 7)` |
| `random.randint` vs `np.integers` | one includes the top, one does not | check which library you are in |
| assumed uniform without looking | same range and mean, wrong shape | histogram it, then `chisquare` or `kstest` |
| "the test found nothing, so it is fair" | 60 rolls miss a blatant cheat 6 times in 10 | say how much data you had |
| used `np.random` for a token | the sequence is predictable | the `secrets` module |

In [ ]:
#  proof that three of those really do fail
print("randint(1,6).pmf(6) =", stats.randint(1, 6).pmf(6), " <- a 5-sided die")
print("uniform(5,10) spans  ", stats.uniform(5, 10).ppf([0, 1]), " <- U(5,15)")
print("uniform(0,0.1).pdf   =", stats.uniform(0, 0.1).pdf(0.05), " <- a density above 1")
print("P(X = 4.7) exactly   =", stats.uniform(0, 10).cdf(4.7) - stats.uniform(0, 10).cdf(4.7))

---
# Recap — today in twelve lines

1. Uniform means every outcome equally likely — no peak, no skew.
2. Discrete: n outcomes, each 1/n. Mean (a+b)/2, variance (n²−1)/12.
3. Continuous: a flat rectangle of height 1/(b−a) over a to b.
4. ⭐ **Probability is AREA**, not height. Width × height.
5. So P(X = any exact value) = 0 — which is not the same as impossible.
6. ⚠️ A density can exceed 1. U(0, 0.1) has height 10, and area 1.
7. Continuous mean (a+b)/2, variance (b−a)²/12. sd of U(0,1) is 0.2887.
8. ⚠️ scipy takes loc and **width**: U(a,b) is `uniform(a, b−a)`.
9. Same range and same mean do **not** make it uniform. Look at the shape.
10. Test it: `chisquare` for discrete, `kstest` for continuous — and say your n.
11. Every random number starts as U(0,1); `ppf` reshapes it into anything.
12. One uniform is flat. Twelve added together are a bell curve.

### Homework
- Build a sampler for a distribution scipy does not have, using only `rng.random()`.
- Take any numeric column from your Day 21 project and test whether its leading digits follow
  Benford.
- Find one place in your own code where you assumed uniform without checking.

### Next class — Topic 2.7: The Central Limit Theorem
Three times now a distribution has turned into a bell curve for no apparent reason. Next class
is the one that explains it — and it is the result the whole of the rest of statistics rests on.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*